# LTSR — Colab 実行ノートブック（軽量版）

**層選択型 Transformer シンボリック回帰**の D-アクション改善版パイプライン。
**無料 Colab 前提**で、軽量デフォルト＋各フェーズ後に **Drive 自動保存**（切断対策）にしてあります。

> **Colab に PowerShell はありません**（Linux+Jupyter）。セルの `!コマンド`(bash) で実行します。

### 重要な前提（ハマりどころ）
- **パイプラインは `!python scripts/...`（subprocess）で走る** → `!pip`/`!python` が解決する Python に依存が入っていればOK。
  （カーネルが 3.12、subprocess が 3.11 のように**分離していても動く**。判定は subprocess テストで行う）
- NeSymReS は古い `hydra-core` に依存し 3.11/3.12 で import エラーになるため、手順4で **hydra を 1.3.2 に更新**して回避。
- 重みは **wget で直接取得**（HF クライアントは未認証だと遅い/Xet 化で hf_transfer 無効）。
- **`MPLBACKEND=Agg`** を設定（subprocess 内の matplotlib inline backend クラッシュ回避）。
- **先に GitHub へ push**：このノートは repo を clone します。新スクリプトを push 済みにしてから実行（未 push は付録A の Drive 方式）。

### 実行順
1. clone → 2. 依存 → 3. 重み → 4. hydra更新 → 5. 動作確認 → 6. Drive マウント
→ 7-1 スイート生成 → 7-2 Phase4(CI) → 7-3 Phase5 → 7-4 Phase6(H3) → 7-5 Phase8(LODO)

各フェーズは**軽量デフォルト**（`--eval-limit` 等で decode を削減）。精度重視なら値を上げてください。

## 0. ランタイム確認（GPU 必須）

上部メニュー『ランタイム > ランタイムのタイプを変更 > GPU (T4)』を選択済みか確認。

In [ ]:
import sys
print('KERNEL python:', sys.version.split()[0])
import subprocess
print('subprocess python:', subprocess.run(['python','-c','import sys;print(sys.version.split()[0])'],
      capture_output=True, text=True).stdout.strip(), '(← !python が使う版。ここに依存が入ればOK)')

## 1. リポジトリ取得

private の場合 `REPO_URL` を `https://<TOKEN>@github.com/...` に。別ブランチに push したら `BRANCH` を変更。

In [ ]:
REPO_URL = "https://github.com/blabo25226/Layer-selective_Transformer-based_Symbolic_Regression.git"
BRANCH   = "main"
%cd /content
import os
if not os.path.isdir('/content/LTSR'):
    !git clone --branch $BRANCH $REPO_URL LTSR
%cd /content/LTSR
!git pull --ff-only || true
!ls scripts/ | grep -E 'phase4_multiseed|phase6_noise_sweep|phase8_lodo|generate_diverse_suite' \
  || echo '!! 新スクリプトが無い → ローカルで git push してから git pull'

## 2. 依存関係のインストール

torch は Colab 既存を使う（固定しない）。nesymres は `--no-deps` で入れ、古い pin(hydra1.0/torch2.5) 衝突を回避。

In [ ]:
%cd /content/LTSR
# nesymres 実行に必要な依存（torch は触らない）
!pip install -q 'pytorch-lightning==1.9.5' 'sympy==1.13.1' 'omegaconf==2.3.0' \
    'hydra-core==1.3.2' tensorboardX sympytorch zss 'gym==0.26.2' scikit-learn
# nesymres 本体（依存は上で導入済み → pin 衝突回避）
!pip install -q -e NSRS/src --no-deps
print('deps installed')

## 3. NeSymReS 重み（約317MB, wget 直取得）

HF クライアントは未認証で遅いので wget で CDN から直接取得（認証不要）。TPSR も同じ重みを共有。

In [ ]:
import os
os.makedirs('NSRS/weights', exist_ok=True)
if not os.path.exists('NSRS/weights/10M.ckpt'):
    !wget -q --show-progress -O NSRS/weights/10M.ckpt \
      https://huggingface.co/TommasoBendinelli/NeuralSymbolicRegressionThatScales/resolve/main/10M.ckpt
!mkdir -p TPSR/nesymres/weights && ln -sf $(realpath NSRS/weights/10M.ckpt) TPSR/nesymres/weights/10M.ckpt
print('weights:', os.path.getsize('NSRS/weights/10M.ckpt')//10**6, 'MB')

## 4. 動作確認（subprocess で import）

**カーネル内 import ではなく `!python`（＝パイプラインと同じ subprocess）で確認**します。
`OK nesymres` が出れば準備完了。`mutable default` エラーが出たら手順2の hydra 更新を再実行。

In [ ]:
import os; os.environ['MPLBACKEND'] = 'Agg'   # 以後の !python に継承
!MPLBACKEND=Agg python -c "import sys; sys.path.insert(0,'NSRS/src'); \
  from nesymres.architectures.model import Model; print('OK nesymres')"

## 5. Google Drive マウント（自動保存先）

無料 Colab は切断で `/content` が消えるため、各フェーズ後に Drive へ保存します。

In [ ]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/LTSR_results', exist_ok=True)
print('保存先:', '/content/drive/MyDrive/LTSR_results')

---
# パイプライン（軽量デフォルト・各セルで Drive 自動保存）

**各セルは順に実行**。重いセルは末尾で `results/` を Drive にコピーします（切断してもそこまでは残る）。
時間の目安は T4/無料枠での概算。`--eval-limit` を上げると精度↑・時間↑。

## 7-1. 合成スイート生成（多骨格・構造分割 + ノイズ2水準） 〜1分

軽量化のため `n-per-skeleton=6`（train~60/test~30）、ノイズは 0.0/0.1 の2水準。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
!python scripts/generate_diverse_suite.py --n-per-skeleton 6 --noise 0.0 0.1
!cat results/synthetic/diverse_v1/suite_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

## 7-2. Phase 4 multi-seed（層寄与 CI） 〜15-20分

3 seed・`eval-limit 10`・短縮 BFGS。層寄与の mean±95%CI と top-3 安定度を出力。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
!python scripts/phase4_multiseed.py --data-dir results/synthetic/diverse_v1 \
    --seeds 0 1 2 --epochs 3 --eval-limit 10 --bfgs-stop-time 0.3
!echo '===== report ====='; cat results/phase_results/phase4_multiseed_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

## 7-3. Phase 5 選択的 FT（動的ランキング + 健全 random 対照） 〜10-15分

Phase 4 の `contributions.json`（単一seed）からランキングを動的取得。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
# 動的ランキング用に単一seedの Phase4 出力（contributions.json）を先に作る
!python scripts/phase4_layer_contribution.py --data-dir results/synthetic/diverse_v1 \
    --epochs 3 --eval-limit 10 --bfgs-stop-time 0.3
!python scripts/phase5_selective_train.py --data-dir results/synthetic/diverse_v1 \
    --epochs 5 --eval-limit 10 --bfgs-stop-time 0.3
!echo '===== report ====='; cat results/phase_results/phase5_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

## 7-4. Phase 6 ノイズスイープ（H3） 〜15-20分

ノイズ 0.0/0.1、TPSR は軽量（rollout=2,width=2）、`eval-limit 8`。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
!python scripts/phase6_noise_sweep.py --noise 0.0 0.1 --epochs 5 \
    --eval-limit 8 --rollout 2 --width 2
!echo '===== report ====='; cat results/phase_results/phase6_noise_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

## 7-5. Phase 8 LODO（donor 交差検証） 〜10分

まず PySR 無しで（速い）。PySR baseline は次セル（任意・低速）。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
!python scripts/phase8_lodo.py --epochs 5
!echo '===== report ====='; cat results/phase_results/phase8_lodo_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

### 7-5b.（任意）PySR baseline 込み LODO 〜20-40分

Julia のインストールが必要。時間に余裕があるときだけ。

In [ ]:
import os; os.environ['MPLBACKEND']='Agg'
!pip install -q pysr
!python scripts/phase8_lodo.py --epochs 5 --with-pysr --pysr-iters 12
!echo '===== report ====='; cat results/phase_results/phase8_lodo_report.md
# --- 自動 Drive 保存（切断対策）---
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ 2>/dev/null && echo '💾 saved to Drive' || echo '⚠️ Drive未マウント → 手順6を実行'

## 8. レポートまとめ表示

In [ ]:
import os
from IPython.display import Markdown, display
for r in ['phase4_multiseed_report.md','phase5_report.md','phase6_noise_report.md','phase8_lodo_report.md']:
    p = f'results/phase_results/{r}'
    if os.path.exists(p):
        display(Markdown('# 📄 '+r+'\n\n'+open(p, encoding='utf-8').read()))
    else:
        print('(missing)', p)

## 9. 最終 Drive 保存

In [ ]:
!cp -ru results/* /content/drive/MyDrive/LTSR_results/ && echo 'done: /content/drive/MyDrive/LTSR_results'

---
## 付録A: push していない / private の場合（Drive から実行）

1. ローカルの `LTSR` を Google Drive の `MyDrive/LTSR` にアップロード（重みは手順3で取得するので除外可）。
2. `from google.colab import drive; drive.mount('/content/drive')` → `%cd /content/drive/MyDrive/LTSR`。
3. 手順2(依存)・3(重み)・4(確認)・6(マウント)・7(パイプライン) を同様に実行。

## 付録B: トラブルシュート

| 症状 | 対処 |
|------|------|
| `No module named 'pytorch_lightning'` 等（カーネル内 import） | それは 3.12 カーネルの制限。**パイプラインは `!python`(subprocess) で動く**ので手順4の subprocess テストで判定。カーネル内 import は不要 |
| `mutable default ... override_dirname` | 手順2の `hydra-core==1.3.2` を再実行 |
| `matplotlib ... backend` (subprocess) | 各セル先頭の `os.environ['MPLBACKEND']='Agg'` を実行。または `!MPLBACKEND=Agg python ...` |
| 重みDLが遅い | wget（手順3）を使用。HF クライアント/hf_transfer は使わない |
| 実行が長すぎ/切断が怖い | `--eval-limit` を下げる、`--noise` を1水準、`--seeds 0 1`、PySR を外す |
| GPUに接続できません | 無料枠のクォータ切れ。時間を置く |
| 途中切断 | Drive 保存済みの `results/` から再開。スクリプトに resume は無いので未完フェーズは再実行 |

## 付録C: (最終手段) condacolab で Python 3.10/3.11

手順2の hydra 更新でも subprocess import が通らない場合のみ。**実行後 `The Kernel crashed` は正常**（意図的な再起動）。
再起動後は condacolab セルを再実行せず手順1から続行。

In [ ]:
!pip install -q condacolab
import condacolab; condacolab.install()  # 再起動（'Kernel crashed' 表示は正常）